# 01a Parse MET Nordic Analysis - Realised Weather Daily and Windows

This notebook converts raw MET Nordic Analysis NetCDF files into canonical realised-weather panels. It writes the existing UTC-day daily proxy and a Europe/Oslo local-time window panel for realised-weather window comparison.


## Pipeline position

This parser runs before the main 01-07 pipeline. Its outputs provide realised-weather inputs for ex-post statistics, validation, calibration diagnostics, and the M3 perfect-information benchmark. The weather source is a MET Nordic Analysis proxy, not station observations.

Inputs are the raw `analysis_temp_nc`, `analysis_precip_nc`, `analysis_wind_nc`, `analysis_humid_nc`, and `analysis_cloud_nc` folders under the FIMEX output root, `cloud_points.cfg`, and `thesis_context/Avd_lokasjon.xlsx`. When executed, the notebook writes `data/processed/realised_weather_daily.parquet` keyed by `(date, AvdelingID)` and `data/processed/realised_weather_daily_windows.parquet` keyed by `(date, AvdelingID, aggregation_window)`. The window output includes `full_day_00_23`, `trade_08_18`, and `trade_08_19`. The full-day window is retained as a benchmark; trading-window use should follow coverage and sales-profile validation.


## Imports and constants

The setup defines project-root discovery, raw-data paths, MET variable mappings, output locations, physical bounds, and local-time window specifications.


In [ ]:
import gc
import os
import re
import time
from datetime import date, timedelta
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import netCDF4
except ImportError as exc:
    raise ImportError(
        "netCDF4 is required for this parser. Install with `mamba install -n csvi_env netCDF4` "
        "or `pip install netCDF4` and restart the kernel."
    ) from exc

def find_project_root(start: Path | None = None) -> Path:
    """Find the repository root from marker files, starting at cwd by default."""
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "README_FOR_CODEX.md").exists() and (candidate / "AGENTS.md").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate project root from README_FOR_CODEX.md and AGENTS.md. "
        "Start Jupyter from the repository root."
    )


def resolve_raw_data_root() -> Path:
    """Resolve the external FIMEX output folder without creating alternatives."""
    candidates = []
    env_value = os.environ.get("FIMEX_OUT_ROOT")
    if env_value:
        candidates.append(Path(env_value))
    candidates.extend([
        Path("C:/Users/<USER>/fimex_out"),
        Path("/mnt/c/Users/<USER>/fimex_out"),
    ])
    for candidate in candidates:
        if candidate.exists():
            return candidate
    expected = " or ".join(str(p) for p in candidates)
    raise FileNotFoundError(f"Missing raw NetCDF root. Expected one of: {expected}")


PROJECT_ROOT = find_project_root()
RAW_DATA_ROOT = resolve_raw_data_root()
AVD_LOKASJON = PROJECT_ROOT / "thesis_context" / "Avd_lokasjon.xlsx"
CFG_PATH = RAW_DATA_ROOT / "cloud_points.cfg"
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "realised_weather_daily.parquet"
WINDOW_OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "realised_weather_daily_windows.parquet"
LOCAL_TIMEZONE = "Europe/Oslo"

# Map each weather variant to its raw folder, NetCDF variable, and filename suffix.
VARIANTS = {
    "temp":   ("analysis_temp_nc",   "air_temperature_2m",   "temp"),
    "precip": ("analysis_precip_nc", "precipitation_amount", "precip"),
    "wind":   ("analysis_wind_nc",   "wind_speed_10m",       "wind"),
    "humid":  ("analysis_humid_nc",  "relative_humidity_2m", "humid"),
    "cloud":  ("analysis_cloud_nc",  "cloud_area_fraction",  "cloud"),
}

WINDOW_DEFINITIONS = {
    "full_day_00_23": {"start_hour": 0, "end_hour": 23, "expected_hours": 24},
    "trade_08_18": {"start_hour": 8, "end_hour": 18, "expected_hours": 11},
    "trade_08_19": {"start_hour": 8, "end_hour": 19, "expected_hours": 12},
}
WINDOW_ORDER = list(WINDOW_DEFINITIONS.keys())
WINDOW_EXPECTED_HOURS = {name: spec["expected_hours"] for name, spec in WINDOW_DEFINITIONS.items()}


def compute_expected_hours_for_window(local_date, aggregation_window, tz=LOCAL_TIMEZONE) -> int:
    """Return DST-aware expected local civil hours for a window."""
    window_name = str(aggregation_window)
    if window_name == "trade_08_18":
        return 11
    if window_name == "trade_08_19":
        return 12
    if window_name != "full_day_00_23":
        raise ValueError(f"Unknown aggregation_window: {aggregation_window}")

    local_day = pd.Timestamp(local_date).normalize()
    local_start = local_day.tz_localize(tz)
    local_end = (local_day + pd.Timedelta(days=1)).tz_localize(tz)
    delta = local_end.tz_convert("UTC") - local_start.tz_convert("UTC")
    return int(delta / pd.Timedelta(hours=1))


def assign_expected_hours_for_windows(frame: pd.DataFrame, date_col: str) -> pd.Series:
    """Vector-friendly wrapper for DST-aware expected window hours."""
    unique_pairs = frame[[date_col, "aggregation_window"]].drop_duplicates()
    expected_map = {
        (
            pd.Timestamp(row[date_col]).normalize(),
            str(row["aggregation_window"]),
        ): compute_expected_hours_for_window(
            row[date_col], row["aggregation_window"]
        )
        for _, row in unique_pairs.iterrows()
    }
    keys = zip(pd.to_datetime(frame[date_col]).dt.normalize(), frame["aggregation_window"].astype(str))
    return pd.Series([expected_map[key] for key in keys], index=frame.index, dtype="int8")

LAT_LON_TOLERANCE = 0.001  # degrees
LOG_EVERY = 100            # progress message cadence, per project convention
SENTINEL_ABS_THRESHOLD = 1e6

# Fail early on missing raw inputs; raw folders are never created here.
for path, label in [
    (RAW_DATA_ROOT, "raw data root"),
    (AVD_LOKASJON, "Avd_lokasjon.xlsx"),
    (CFG_PATH, "cloud_points.cfg"),
]:
    if not path.exists():
        raise FileNotFoundError(f"Missing {label}: {path}")

for variant, (sub_folder, _ncvar, _suffix) in VARIANTS.items():
    folder = RAW_DATA_ROOT / sub_folder
    if not folder.exists():
        raise FileNotFoundError(f"Missing NetCDF folder for {variant}: {folder}")

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data root: {RAW_DATA_ROOT}")
print(f"Daily output: {OUTPUT_PATH}")
print(f"Window output: {WINDOW_OUTPUT_PATH}")
print(f"Local-time aggregation timezone: {LOCAL_TIMEZONE}")


## Store mapping

The next step reads `Avd_lokasjon.xlsx` and `cloud_points.cfg`, then verifies that the store latitude/longitude order matches the NetCDF point order within `0.001` degrees. This alignment links each NetCDF `x` index to the correct `AvdelingID`.


In [ ]:
def parse_cloud_points_cfg(cfg_path: Path) -> tuple[np.ndarray, np.ndarray]:
    """Parse latitudeValues and longitudeValues from a FIMEX nearest-neighbor cfg."""
    text = cfg_path.read_text(encoding="utf-8", errors="replace")
    lat_match = re.search(r"^\s*latitudeValues\s*=\s*([^\n]+)$", text, flags=re.MULTILINE)
    lon_match = re.search(r"^\s*longitudeValues\s*=\s*([^\n]+)$", text, flags=re.MULTILINE)
    if not lat_match or not lon_match:
        raise ValueError(f"Could not locate latitudeValues / longitudeValues in {cfg_path}")
    lats = np.array([float(x.strip()) for x in lat_match.group(1).split(",") if x.strip()])
    lons = np.array([float(x.strip()) for x in lon_match.group(1).split(",") if x.strip()])
    return lats, lons


cfg_lats, cfg_lons = parse_cloud_points_cfg(CFG_PATH)
avd_df = pd.read_excel(AVD_LOKASJON)

# Resolve store identifier and coordinate columns across source-file naming variants.
def _find_col(frame, candidates):
    for c in candidates:
        if c in frame.columns:
            return c
    raise KeyError(f"None of {candidates} found in columns {list(frame.columns)}")


col_avd = _find_col(avd_df, ["AvdelingID", "Avdeling", "id"])
col_lat = _find_col(avd_df, ["Lat", "latitude", "Latitude", "lat"])
col_lon = _find_col(avd_df, ["Lon", "longitude", "Longitude", "lon", "Lng"])

if len(avd_df) != len(cfg_lats):
    raise ValueError(
        f"Length mismatch: Avd_lokasjon.xlsx has {len(avd_df)} rows but cfg has "
        f"{len(cfg_lats)} latitude entries."
    )
if len(cfg_lats) != len(cfg_lons):
    raise ValueError(
        f"cfg latitudeValues / longitudeValues length mismatch: {len(cfg_lats)} vs {len(cfg_lons)}"
    )

excel_lats = avd_df[col_lat].to_numpy(dtype="float64")
excel_lons = avd_df[col_lon].to_numpy(dtype="float64")

lat_diff = np.abs(excel_lats - cfg_lats)
lon_diff = np.abs(excel_lons - cfg_lons)
lat_max = float(lat_diff.max())
lon_max = float(lon_diff.max())
lat_bad = int((lat_diff > LAT_LON_TOLERANCE).sum())
lon_bad = int((lon_diff > LAT_LON_TOLERANCE).sum())

if lat_bad > 0 or lon_bad > 0:
    raise AssertionError(
        f"Store-mapping validation failed: {lat_bad} latitudes and {lon_bad} longitudes "
        f"exceed tolerance {LAT_LON_TOLERANCE}. "
        f"Max lat diff = {lat_max:.6f}, max lon diff = {lon_max:.6f}. "
        "Order of Avd_lokasjon.xlsx and cloud_points.cfg may not match."
    )

# Ordered store IDs define the NetCDF x-index to AvdelingID mapping.
AVDELING_BY_X = avd_df[col_avd].astype("int64").to_numpy()
N_STORES = len(AVDELING_BY_X)
assert N_STORES == 46, f"Expected 46 stores, got {N_STORES}"

print(f"Stores validated: {N_STORES}")
print(f"Max lat diff: {lat_max:.6g} (tolerance {LAT_LON_TOLERANCE})")
print(f"Max lon diff: {lon_max:.6g} (tolerance {LAT_LON_TOLERANCE})")
print(f"AvdelingID range: {AVDELING_BY_X.min()}..{AVDELING_BY_X.max()}")
print(avd_df[[col_avd, col_lat, col_lon]].head(3))


## File index

The file index parses the date and hour from each NetCDF filename by weather variable. It reports coverage by variant and identifies missing `(date, hour)` slots relative to a complete 24-hour day.


In [ ]:
FILENAME_RE = re.compile(r"^analysis_(\d{8})T(\d{2})Z_(?P<suffix>[a-zA-Z0-9]+)\.nc$")


def build_file_index(variant: str, sub_folder: str, suffix: str) -> pd.DataFrame:
    folder = RAW_DATA_ROOT / sub_folder
    if not folder.exists():
        raise FileNotFoundError(f"Variant folder missing: {folder}")
    rows = []
    skipped = 0
    for path in folder.iterdir():
        if not path.is_file() or path.suffix != ".nc":
            continue
        m = FILENAME_RE.match(path.name)
        if m is None:
            skipped += 1
            continue
        if m.group("suffix") != suffix:
            skipped += 1
            continue
        ymd = m.group(1)
        hh = int(m.group(2))
        d = date(int(ymd[:4]), int(ymd[4:6]), int(ymd[6:8]))
        rows.append((variant, d, hh, str(path)))
    df = pd.DataFrame(rows, columns=["variant", "date", "hour", "filepath"])
    df = df.sort_values(["date", "hour"]).reset_index(drop=True)
    return df


variant_indexes = {}
for variant, (sub, _ncvar, suffix) in VARIANTS.items():
    idx = build_file_index(variant, sub, suffix)
    variant_indexes[variant] = idx
    n_files = len(idx)
    n_dates = idx["date"].nunique() if n_files > 0 else 0
    if n_dates > 0:
        d_min, d_max = idx["date"].min(), idx["date"].max()
        expected_slots = (d_max - d_min + timedelta(days=1)).days * 24
        missing_slots = expected_slots - n_files
    else:
        d_min = d_max = None
        expected_slots = missing_slots = 0
    print(
        f"{variant:7s} files={n_files:>6,} dates={n_dates:>5} "
        f"range={d_min}..{d_max}  missing_slots~={missing_slots:,}"
    )


## Hourly parser

For each weather variable, the parser reads all available hourly files, extracts the 46-store slice, applies unit conversions, and builds an hourly table keyed by `(variant, date, hour, AvdelingID)`. Each variant is processed separately to limit memory use.

Unit conversions are Kelvin to Celsius for `temp`, fractions to percentages for `humid` and `cloud`, and native units for `precip` and `wind`.


In [ ]:
UNIT_CONVERTERS = {
    "temp":   lambda arr: arr.astype("float32") - 273.15,
    "precip": lambda arr: arr.astype("float32"),
    "wind":   lambda arr: arr.astype("float32"),
    "humid":  lambda arr: np.clip(arr.astype("float32") * 100.0, 0.0, 100.0),
    "cloud":  lambda arr: np.clip(arr.astype("float32") * 100.0, 0.0, 100.0),
}

variant_metadata_log = {}


def read_netcdf_vector(ds: netCDF4.Dataset, ncvar: str, expected_size: int) -> np.ndarray:
    """Read one NetCDF variable, masking fill values and defensive sentinels to NaN."""
    raw = ds.variables[ncvar][:]
    if hasattr(raw, "filled"):
        raw = raw.filled(np.nan)
    vals = np.asarray(raw, dtype="float32").flatten()
    if vals.size != expected_size:
        raise ValueError(f"Unexpected payload size {vals.size}; expected {expected_size}")
    vals = np.where(np.abs(vals) > SENTINEL_ABS_THRESHOLD, np.nan, vals).astype("float32")
    return vals


def log_variant_metadata_once(variant: str, ds: netCDF4.Dataset, ncvar: str) -> None:
    """Print metadata once per variable, including the realised precipitation treatment."""
    if variant in variant_metadata_log:
        return
    var = ds.variables[ncvar]
    attrs = {
        "units": getattr(var, "units", ""),
        "standard_name": getattr(var, "standard_name", ""),
        "long_name": getattr(var, "long_name", ""),
    }
    variant_metadata_log[variant] = attrs
    print(
        f"    {variant}: NetCDF metadata units={attrs['units']!r}, "
        f"standard_name={attrs['standard_name']!r}, long_name={attrs['long_name']!r}"
    )
    if variant == "precip":
        print(
            "    precip: treating MET Nordic Analysis precipitation_amount as hourly amount; "
            "no cumulative differencing is applied in 01a."
        )


def add_local_time_columns(hourly_df: pd.DataFrame) -> pd.DataFrame:
    """Add UTC datetime and Europe/Oslo local date/hour columns used by window aggregates."""
    if hourly_df.empty:
        return hourly_df
    hourly_df["datetime_utc"] = hourly_df["date"] + pd.to_timedelta(
        hourly_df["hour"].astype("int16"), unit="h"
    )
    dt_utc = pd.to_datetime(hourly_df["datetime_utc"]).dt.tz_localize("UTC")
    dt_local = dt_utc.dt.tz_convert(LOCAL_TIMEZONE)
    hourly_df["date_local"] = dt_local.dt.tz_localize(None).dt.normalize()
    hourly_df["hour_local"] = dt_local.dt.hour.astype("int8")
    hourly_df["hour_utc"] = hourly_df["hour"].astype("int8")
    return hourly_df


def parse_one_variant(variant: str) -> tuple[pd.DataFrame, list[str]]:
    """Read every NetCDF file for a variant; return (hourly_df, corrupt_paths)."""
    idx = variant_indexes[variant]
    _sub, ncvar, _suffix = VARIANTS[variant]
    converter = UNIT_CONVERTERS[variant]
    n_files = len(idx)
    if n_files == 0:
        return pd.DataFrame(columns=["date", "hour", "AvdelingID", "value"]), []

    # Pre-allocate flat arrays to avoid repeated row-wise appends.
    total_rows = n_files * N_STORES
    dates_arr = np.empty(total_rows, dtype="datetime64[D]")
    hours_arr = np.empty(total_rows, dtype="int8")
    avd_arr = np.empty(total_rows, dtype="int64")
    val_arr = np.empty(total_rows, dtype="float32")

    corrupt_paths: list[str] = []
    cursor = 0
    t0 = time.time()
    for i, row in enumerate(idx.itertuples(index=False)):
        try:
            with netCDF4.Dataset(row.filepath, "r") as ds:
                log_variant_metadata_once(variant, ds, ncvar)
                vals = read_netcdf_vector(ds, ncvar, N_STORES)
                vals = converter(vals)
        except Exception as exc:
            corrupt_paths.append(f"{row.filepath} :: {type(exc).__name__}: {exc}")
            continue
        dates_arr[cursor:cursor + N_STORES] = np.datetime64(row.date)
        hours_arr[cursor:cursor + N_STORES] = row.hour
        avd_arr[cursor:cursor + N_STORES] = AVDELING_BY_X
        val_arr[cursor:cursor + N_STORES] = vals
        cursor += N_STORES
        if (i + 1) % LOG_EVERY == 0:
            elapsed = time.time() - t0
            rate = (i + 1) / max(elapsed, 1e-6)
            print(f"    {variant}: parsed {i + 1:>6,} / {n_files:,} files ({rate:.0f} files/s)")

    hourly_df = pd.DataFrame({
        "date": pd.to_datetime(dates_arr[:cursor]),
        "hour": hours_arr[:cursor],
        "AvdelingID": avd_arr[:cursor],
        "value": val_arr[:cursor],
    })
    hourly_df["date"] = hourly_df["date"].dt.normalize()
    hourly_df = add_local_time_columns(hourly_df)
    return hourly_df, corrupt_paths


## Daily aggregation

The hourly data are aggregated to `(date, AvdelingID)` using mean, minimum, maximum, or sum according to the weather variable. `hours_available` records the number of hourly observations behind each daily cell.


In [ ]:
AGGREGATION_SPECS = {
    "temp":   {"mean": ("value", "mean"),  "min": ("value", "min"),  "max": ("value", "max")},
    "precip": {"sum":  ("value", "sum")},
    "wind":   {"mean": ("value", "mean"),  "max": ("value", "max")},
    "humid":  {"mean": ("value", "mean")},
    "cloud":  {"mean": ("value", "mean")},
}


def aggregate_to_daily(hourly_df: pd.DataFrame, variant: str) -> pd.DataFrame:
    """Aggregate to the existing UTC-day daily output schema."""
    if hourly_df.empty:
        return pd.DataFrame()
    spec = AGGREGATION_SPECS[variant]
    agg_kwargs = {f"{variant}_obs_{label}": pair for label, pair in spec.items()}
    daily = (
        hourly_df.groupby(["date", "AvdelingID"], observed=True)
        .agg(hours_available=("value", "count"), **agg_kwargs)
        .reset_index()
    )
    return daily


def aggregate_to_windows(hourly_df: pd.DataFrame, variant: str) -> pd.DataFrame:
    """Aggregate to Europe/Oslo local-time candidate windows for one variant."""
    if hourly_df.empty:
        return pd.DataFrame()
    spec = AGGREGATION_SPECS[variant]
    agg_kwargs = {f"{variant}_obs_{label}": pair for label, pair in spec.items()}
    frames = []
    base_cols = ["date_local", "AvdelingID", "datetime_utc", "hour_local", "hour_utc", "value"]
    for window_name in WINDOW_ORDER:
        window = WINDOW_DEFINITIONS[window_name]
        mask = hourly_df["hour_local"].between(window["start_hour"], window["end_hour"])
        sub = hourly_df.loc[mask, base_cols]
        sub = sub.loc[sub["value"].notna()]
        if sub.empty:
            continue
        sub = sub.sort_values(["date_local", "AvdelingID", "datetime_utc"])
        grouped = (
            sub.groupby(["date_local", "AvdelingID"], observed=True)
            .agg(
                **agg_kwargs,
                hours_in_window=("value", "size"),
                first_hour_local=("hour_local", "min"),
                last_hour_local=("hour_local", "max"),
                first_hour_utc=("hour_utc", "first"),
                last_hour_utc=("hour_utc", "last"),
            )
            .reset_index()
            .rename(columns={"date_local": "date"})
        )
        grouped["aggregation_window"] = window_name
        grouped["expected_hours_in_window"] = window["expected_hours"]
        frames.append(grouped)
    if not frames:
        return pd.DataFrame()
    out = pd.concat(frames, ignore_index=True)
    return out


variant_daily = {}
variant_windows = {}
all_corrupt = []

for variant in VARIANTS:
    print(f"--- parsing {variant} ---")
    t0 = time.time()
    hourly_df, corrupt_paths = parse_one_variant(variant)
    print(f"    {variant}: hourly rows = {len(hourly_df):,}; corrupt files = {len(corrupt_paths)}")
    if corrupt_paths:
        all_corrupt.extend(corrupt_paths)
    daily = aggregate_to_daily(hourly_df, variant)
    windows = aggregate_to_windows(hourly_df, variant)
    print(
        f"    {variant}: daily rows = {len(daily):,}; "
        f"window rows = {len(windows):,}; took {(time.time() - t0) / 60:.2f} min"
    )
    variant_daily[variant] = daily
    variant_windows[variant] = windows
    del hourly_df
    gc.collect()

print(f"\nTotal corrupt-file warnings across variants: {len(all_corrupt)}")
if all_corrupt[:5]:
    print("First 5 corrupt-file entries:")
    for line in all_corrupt[:5]:
        print(f"  {line}")


## Gap handling

The notebook constructs a complete `(date, AvdelingID)` grid for the observed date range and fills missing daily weather cells. Temperature, wind, humidity, and cloud gaps use within-store time interpolation with edge fills; precipitation gaps use a store-level +/-7-day rolling mean, then the store mean if needed. Rows filled from missing source data are flagged with `is_interpolated = True`.


In [ ]:
def build_complete_grid(daily_frames: dict[str, pd.DataFrame]) -> pd.DataFrame:
    # Use the union of variant dates to define the complete daily grid.
    all_dates = pd.Index([], dtype="datetime64[ns]")
    for v, df in daily_frames.items():
        if not df.empty:
            all_dates = all_dates.union(df["date"].unique())
    if len(all_dates) == 0:
        raise RuntimeError("No daily data was produced for any variant.")
    d_min, d_max = all_dates.min(), all_dates.max()
    full_dates = pd.date_range(d_min, d_max, freq="D")
    grid = pd.MultiIndex.from_product(
        [full_dates, AVDELING_BY_X], names=["date", "AvdelingID"]
    ).to_frame(index=False)
    return grid


grid = build_complete_grid(variant_daily)
print(f"Complete grid rows: {len(grid):,} (= {grid['date'].nunique()} days x {N_STORES} stores)")


def fill_variant(grid_df: pd.DataFrame, daily_df: pd.DataFrame, variant: str) -> pd.DataFrame:
    """Left-join daily into the grid, mark gaps, and interpolate per the variant rule."""
    merged = grid_df.merge(daily_df, on=["date", "AvdelingID"], how="left")
    merged["hours_available"] = merged["hours_available"].fillna(0).astype("int16")
    value_columns = [c for c in merged.columns if c.startswith(f"{variant}_obs_")]

    # Sort before interpolation to preserve within-store date order.
    # Compute is_gap after sort/reset so the mask aligns with the merged frame.
    merged = merged.sort_values(["AvdelingID", "date"]).reset_index(drop=True)
    is_gap = merged["hours_available"] == 0
    merged[f"{variant}_is_interpolated"] = is_gap

    if variant == "precip":
        # +/-7-day rolling mean per store for precipitation gaps.
        for col in value_columns:
            rolled = (
                merged.groupby("AvdelingID", observed=True)[col]
                .transform(lambda s: s.rolling(window=15, min_periods=1, center=True).mean())
            )
            merged[col] = merged[col].where(~is_gap, rolled)
            # Store mean fallback covers stores with no valid rolling-window values.
            still_na = merged[col].isna()
            if still_na.any():
                store_mean = merged.groupby("AvdelingID", observed=True)[col].transform("mean")
                merged.loc[still_na, col] = store_mean.loc[still_na]
    else:
        for col in value_columns:
            interp_series = (
                merged.groupby("AvdelingID", observed=True)[col]
                .transform(lambda s: s.interpolate(method="linear", limit_direction="both"))
            )
            merged[col] = merged[col].where(~is_gap, interp_series)
    return merged


variant_filled = {}
for variant, daily_df in variant_daily.items():
    filled = fill_variant(grid, daily_df, variant)
    n_gaps = int(filled[f"{variant}_is_interpolated"].sum())
    print(f"    {variant}: gap-filled rows = {n_gaps:,} / {len(filled):,}")
    variant_filled[variant] = filled
    del daily_df
gc.collect()


## Variable merge

The filled variable-level frames are outer-joined on `(date, AvdelingID)`. The merged `hours_available` is the minimum across variables, and `is_interpolated` is true when any weather variable was interpolated.


In [ ]:
def select_variant_columns(df: pd.DataFrame, variant: str) -> pd.DataFrame:
    keep = ["date", "AvdelingID", "hours_available"]
    keep += [c for c in df.columns if c.startswith(f"{variant}_obs_")]
    keep += [f"{variant}_is_interpolated"]
    out = df[keep].rename(columns={"hours_available": f"{variant}_hours_available"})
    return out


merged = None
for variant in VARIANTS:
    sub = select_variant_columns(variant_filled[variant], variant)
    if merged is None:
        merged = sub
    else:
        merged = merged.merge(sub, on=["date", "AvdelingID"], how="outer")

# A row is fully observed only to the minimum coverage across variables.
hours_cols = [f"{v}_hours_available" for v in VARIANTS]
interp_cols = [f"{v}_is_interpolated" for v in VARIANTS]

merged["hours_available"] = merged[hours_cols].min(axis=1).astype("int16")
merged["is_interpolated"] = merged[interp_cols].any(axis=1)

merged = merged.drop(columns=hours_cols + interp_cols)

# Preserve the published daily schema order.
ordered_cols = [
    "date", "AvdelingID",
    "temp_obs_mean", "temp_obs_min", "temp_obs_max",
    "precip_obs_sum",
    "wind_obs_mean", "wind_obs_max",
    "humid_obs_mean",
    "cloud_obs_mean",
    "hours_available", "is_interpolated",
]
missing = [c for c in ordered_cols if c not in merged.columns]
if missing:
    raise AssertionError(f"Final frame missing columns: {missing}")
merged = merged[ordered_cols].sort_values(["date", "AvdelingID"]).reset_index(drop=True)

dup = int(merged.duplicated(["date", "AvdelingID"]).sum())
if dup != 0:
    raise AssertionError(f"Final frame has {dup} duplicate (date, AvdelingID) keys.")

print(f"Final frame rows: {len(merged):,}")
print(f"Columns: {list(merged.columns)}")
print(f"Memory (approx): {merged.memory_usage(deep=True).sum()/1_000_000:.1f} MB")
display(merged.head(5))

# Write an early checkpoint before downstream diagnostics.
merged_for_checkpoint = merged.copy()
merged_for_checkpoint["date"] = merged_for_checkpoint["date"].dt.normalize()
merged_for_checkpoint.to_parquet(OUTPUT_PATH, index=False, engine="pyarrow", compression="snappy")
print(f"Early-checkpoint parquet written: {OUTPUT_PATH} "
      f"({OUTPUT_PATH.stat().st_size/1_000_000:.2f} MB)")
del merged_for_checkpoint


## Local-time window output

This step merges Europe/Oslo window aggregates across variables and writes the long-format `realised_weather_daily_windows.parquet` checkpoint. The existing UTC-day daily output keeps its current schema.


In [ ]:
WINDOW_KEY_COLS = ["date", "AvdelingID", "aggregation_window"]


def build_window_grid(window_frames: dict[str, pd.DataFrame]) -> pd.DataFrame:
    all_dates = pd.Index([], dtype="datetime64[ns]")
    for df in window_frames.values():
        if not df.empty:
            all_dates = all_dates.union(pd.DatetimeIndex(pd.to_datetime(df["date"]).unique()))
    if len(all_dates) == 0:
        raise RuntimeError("No local-time window aggregates were produced for any variant.")
    full_dates = pd.date_range(all_dates.min(), all_dates.max(), freq="D")
    grid = pd.MultiIndex.from_product(
        [full_dates, AVDELING_BY_X, WINDOW_ORDER],
        names=["date", "AvdelingID", "aggregation_window"],
    ).to_frame(index=False)
    grid["expected_hours_in_window"] = assign_expected_hours_for_windows(grid, "date")
    return grid


def select_window_columns(df: pd.DataFrame, variant: str) -> pd.DataFrame:
    value_cols = [c for c in df.columns if c.startswith(f"{variant}_obs_")]
    keep = WINDOW_KEY_COLS + value_cols + [
        "hours_in_window", "first_hour_local", "last_hour_local", "first_hour_utc", "last_hour_utc",
    ]
    out = df[keep].rename(columns={
        "hours_in_window": f"{variant}_hours_in_window",
        "first_hour_local": f"{variant}_first_hour_local",
        "last_hour_local": f"{variant}_last_hour_local",
        "first_hour_utc": f"{variant}_first_hour_utc",
        "last_hour_utc": f"{variant}_last_hour_utc",
    })
    return out


realised_windows = build_window_grid(variant_windows)
for variant in VARIANTS:
    sub = select_window_columns(variant_windows[variant], variant)
    realised_windows = realised_windows.merge(sub, on=WINDOW_KEY_COLS, how="left")

hours_cols = [f"{v}_hours_in_window" for v in VARIANTS]
first_local_cols = [f"{v}_first_hour_local" for v in VARIANTS]
last_local_cols = [f"{v}_last_hour_local" for v in VARIANTS]
first_utc_cols = [f"{v}_first_hour_utc" for v in VARIANTS]
last_utc_cols = [f"{v}_last_hour_utc" for v in VARIANTS]

realised_windows[hours_cols] = realised_windows[hours_cols].fillna(0)
realised_windows["hours_in_window"] = realised_windows[hours_cols].min(axis=1).astype("int8")
realised_windows["coverage_share"] = np.minimum(
    realised_windows["hours_in_window"].astype("float32") /
    realised_windows["expected_hours_in_window"].astype("float32"),
    1.0,
).astype("float32")
realised_windows["is_full_window"] = (
    realised_windows["hours_in_window"] == realised_windows["expected_hours_in_window"]
).astype("int8")
realised_windows["is_partial_window"] = (1 - realised_windows["is_full_window"]).astype("int8")

realised_windows["first_hour_local"] = (
    realised_windows[first_local_cols].max(axis=1, skipna=True).fillna(-1).astype("int8")
)
realised_windows["last_hour_local"] = (
    realised_windows[last_local_cols].min(axis=1, skipna=True).fillna(-1).astype("int8")
)
realised_windows["first_hour_utc"] = (
    realised_windows[first_utc_cols].max(axis=1, skipna=True).fillna(-1).astype("int8")
)
realised_windows["last_hour_utc"] = (
    realised_windows[last_utc_cols].min(axis=1, skipna=True).fillna(-1).astype("int8")
)

realised_windows = realised_windows.drop(
    columns=hours_cols + first_local_cols + last_local_cols + first_utc_cols + last_utc_cols
)

ordered_window_cols = [
    "date", "AvdelingID", "aggregation_window",
    "temp_obs_mean", "temp_obs_min", "temp_obs_max",
    "precip_obs_sum",
    "wind_obs_mean", "wind_obs_max",
    "humid_obs_mean",
    "cloud_obs_mean",
    "hours_in_window", "expected_hours_in_window", "coverage_share",
    "is_partial_window", "is_full_window",
    "first_hour_local", "last_hour_local",
    "first_hour_utc", "last_hour_utc",
]
missing = [c for c in ordered_window_cols if c not in realised_windows.columns]
if missing:
    raise AssertionError(f"Window frame missing columns: {missing}")

realised_windows = realised_windows[ordered_window_cols].sort_values(WINDOW_KEY_COLS).reset_index(drop=True)
realised_windows["date"] = pd.to_datetime(realised_windows["date"]).dt.normalize()
realised_windows["AvdelingID"] = realised_windows["AvdelingID"].astype("int64")
realised_windows["aggregation_window"] = realised_windows["aggregation_window"].astype("string")

float_cols = [
    "temp_obs_mean", "temp_obs_min", "temp_obs_max", "precip_obs_sum",
    "wind_obs_mean", "wind_obs_max", "humid_obs_mean", "cloud_obs_mean", "coverage_share",
]
for col in float_cols:
    realised_windows[col] = realised_windows[col].astype("float32")

int8_cols = [
    "hours_in_window", "expected_hours_in_window", "is_partial_window", "is_full_window",
    "first_hour_local", "last_hour_local", "first_hour_utc", "last_hour_utc",
]
for col in int8_cols:
    realised_windows[col] = realised_windows[col].astype("int8")

dup = int(realised_windows.duplicated(WINDOW_KEY_COLS).sum())
if dup != 0:
    raise AssertionError(f"Window frame has {dup} duplicate keys on {WINDOW_KEY_COLS}")

realised_windows.to_parquet(WINDOW_OUTPUT_PATH, index=False, engine="pyarrow", compression="snappy")
print(
    f"Window checkpoint parquet written: {WINDOW_OUTPUT_PATH} "
    f"({WINDOW_OUTPUT_PATH.stat().st_size / 1_000_000:.2f} MB)"
)
print(f"Window rows: {len(realised_windows):,}")
display(
    realised_windows.groupby("aggregation_window", observed=True)
    .agg(
        rows=("date", "size"),
        mean_coverage=("coverage_share", "mean"),
        full_share=("is_full_window", "mean"),
    )
    .reset_index()
)


## Sanity checks

The checks validate physical ranges and summarize distributions by variable. Out-of-range values fail the run; suspicious but non-fatal distributions are reported for inspection.


In [ ]:
bounds = {
    "temp_obs_mean":  (-40.0, 40.0),
    "temp_obs_min":   (-50.0, 40.0),
    "temp_obs_max":   (-40.0, 50.0),
    "precip_obs_sum": (0.0, 300.0),
    "wind_obs_mean":  (0.0, 50.0),
    "wind_obs_max":   (0.0, 80.0),
    "humid_obs_mean": (0.0, 100.0),
    "cloud_obs_mean": (0.0, 100.0),
}

for col, (lo, hi) in bounds.items():
    if col not in merged.columns:
        raise KeyError(f"Sanity check missing column: {col}")
    vmin = float(merged[col].min())
    vmax = float(merged[col].max())
    if vmin < lo or vmax > hi:
        raise AssertionError(f"{col} out of range: min={vmin}, max={vmax}, expected [{lo}, {hi}]")

n_total = len(merged)
n_dates = merged["date"].nunique()
n_stores = merged["AvdelingID"].nunique()
n_interp = int(merged["is_interpolated"].sum())

print(f"Total rows: {n_total:,}")
print(f"Distinct dates: {n_dates}")
print(f"Distinct stores: {n_stores} (expected {N_STORES})")
print(f"Interpolated rows: {n_interp:,} ({100 * n_interp / max(n_total, 1):.3f}%)")
print(f"\nDistribution per variable:")
display(merged[list(bounds.keys())].describe(percentiles=[0.25, 0.5, 0.75]).round(3).T)


## Diagnostic plots

The optional diagnostics plot one store-level temperature series, the `hours_available` distribution, and dates affected by interpolation. These plots support parser inspection and are not thesis result figures.


In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=42)
sample_avd = int(rng.choice(AVDELING_BY_X))
sample = merged.loc[merged["AvdelingID"] == sample_avd]

fig, axes = plt.subplots(2, 1, figsize=(12, 7), gridspec_kw={"hspace": 0.4})
axes[0].plot(sample["date"], sample["temp_obs_mean"], lw=0.6)
axes[0].set_title(f"Daily mean temperature at AvdelingID={sample_avd}")
axes[0].set_ylabel("deg C")

axes[1].hist(merged["hours_available"], bins=range(0, 26), align="left", rwidth=0.8)
axes[1].set_title("Distribution of hours_available")
axes[1].set_xlabel("hours")
axes[1].set_ylabel("rows")
plt.show(); plt.close(fig)

interp_dates = merged.loc[merged["is_interpolated"], "date"].drop_duplicates().sort_values()
print(f"\nDates with at least one interpolated row: {len(interp_dates)}")
if len(interp_dates) <= 50:
    for d in interp_dates:
        print(f"  {pd.Timestamp(d).date()}")
else:
    print(f"  (too many to list - first 20: {[pd.Timestamp(d).date() for d in interp_dates.head(20)]})")


## Save output

The final daily realised-weather panel is written after schema cleanup and summary reporting.


In [ ]:
# Store date as a date-level field in the final parquet schema.
merged_to_save = merged.copy()
merged_to_save["date"] = merged_to_save["date"].dt.normalize()
merged_to_save.to_parquet(OUTPUT_PATH, index=False, engine="pyarrow", compression="snappy")

size_mb = OUTPUT_PATH.stat().st_size / 1_000_000
print(f"Saved: {OUTPUT_PATH}")
print(f"Size on disk: {size_mb:.2f} MB")
print(f"Total rows: {len(merged_to_save):,}")
print(f"Schema:")
for col, dt in merged_to_save.dtypes.items():
    print(f"  {col:25s} {dt}")

print(f"\nInterpolation summary:")
print(f"  rows with is_interpolated == True: {int(merged_to_save['is_interpolated'].sum()):,}")
print(
    "  total dates with any interpolation: "
    f"{merged_to_save.loc[merged_to_save['is_interpolated'], 'date'].nunique()}"
)

print("\nSample (head 5):")
display(merged_to_save.head(5))

print("\nDistribution (mean, std, min, 25%, 50%, 75%, max) per variable:")
display(
    merged_to_save[
        ["temp_obs_mean", "temp_obs_min", "temp_obs_max", "precip_obs_sum",
         "wind_obs_mean", "wind_obs_max", "humid_obs_mean", "cloud_obs_mean"]
    ].describe(percentiles=[0.25, 0.5, 0.75]).round(3).T
)


## Realised-weather window validation

After parsing, this validation checks the window parquet, basic physical bounds, and the local `full_day_00_23` window against `realised_weather_daily.parquet`. Small differences are expected because the window output uses Europe/Oslo local time, while the legacy daily file is UTC-day based.


In [ ]:
realised_daily_check = pd.read_parquet(OUTPUT_PATH)
realised_windows_check = pd.read_parquet(WINDOW_OUTPUT_PATH)

realised_daily_check["date"] = pd.to_datetime(realised_daily_check["date"]).dt.normalize()
realised_windows_check["date"] = pd.to_datetime(realised_windows_check["date"]).dt.normalize()

print("Window row counts by aggregation_window:")
display(
    realised_windows_check["aggregation_window"]
    .value_counts()
    .sort_index()
    .rename_axis("aggregation_window")
    .reset_index(name="rows")
)

print("Window date/store coverage:")
print(
    f"  date range: {realised_windows_check['date'].min().date()} "
    f"-> {realised_windows_check['date'].max().date()}"
)
print(f"  distinct stores: {realised_windows_check['AvdelingID'].nunique()} (expected {N_STORES})")
print(f"  duplicate keys: {int(realised_windows_check.duplicated(WINDOW_KEY_COLS).sum())}")

display(
    realised_windows_check.groupby("aggregation_window", observed=True)
    .agg(
        rows=("date", "size"),
        min_hours=("hours_in_window", "min"),
        median_hours=("hours_in_window", "median"),
        max_hours=("hours_in_window", "max"),
        mean_coverage=("coverage_share", "mean"),
        full_share=("is_full_window", "mean"),
    )
    .reset_index()
    .round(4)
)

for col, (lo, hi) in {
    "humid_obs_mean": (0.0, 100.0),
    "cloud_obs_mean": (0.0, 100.0),
    "precip_obs_sum": (0.0, np.inf),
}.items():
    vmin = float(realised_windows_check[col].min())
    vmax = float(realised_windows_check[col].max())
    if vmin < lo or vmax > hi:
        raise AssertionError(f"{col} out of expected range: min={vmin}, max={vmax}")

sentinel_cols = [
    "temp_obs_mean", "temp_obs_min", "temp_obs_max", "precip_obs_sum",
    "wind_obs_mean", "wind_obs_max", "humid_obs_mean", "cloud_obs_mean",
]
if realised_windows_check[sentinel_cols].abs().gt(SENTINEL_ABS_THRESHOLD).any().any():
    raise AssertionError("Sentinel-magnitude values found in realised window output.")

full_day = realised_windows_check.loc[
    realised_windows_check["aggregation_window"] == "full_day_00_23",
    [
        "date", "AvdelingID", "temp_obs_mean", "precip_obs_sum",
        "wind_obs_mean", "humid_obs_mean", "cloud_obs_mean",
    ],
]
comparison = full_day.merge(
    realised_daily_check[
        [
            "date", "AvdelingID", "temp_obs_mean", "precip_obs_sum",
            "wind_obs_mean", "humid_obs_mean", "cloud_obs_mean",
        ]
    ],
    on=["date", "AvdelingID"],
    how="inner",
    suffixes=("_local_full_day", "_utc_daily"),
)
print(f"Joined rows for local full-day vs UTC daily comparison: {len(comparison):,}")

rows = []
for col in ["temp_obs_mean", "precip_obs_sum", "wind_obs_mean", "humid_obs_mean", "cloud_obs_mean"]:
    left = f"{col}_local_full_day"
    right = f"{col}_utc_daily"
    sub = comparison[[left, right]].dropna()
    diff = sub[left].astype("float64") - sub[right].astype("float64")
    rows.append({
        "variable": col,
        "n": int(len(sub)),
        "corr": float(sub[left].corr(sub[right])) if len(sub) > 1 else np.nan,
        "mean_diff_local_minus_utc": float(diff.mean()) if len(sub) else np.nan,
        "mean_abs_diff": float(diff.abs().mean()) if len(sub) else np.nan,
        "max_abs_diff": float(diff.abs().max()) if len(sub) else np.nan,
    })
display(pd.DataFrame(rows).round(5))


## Optional sales-window check

This disabled-by-default diagnostic scans `data/Dataset.parquet` to document the sales-hour motivation for candidate trading windows. It should only be enabled intentionally because the transaction source is large.


In [ ]:
RUN_OPTIONAL_SALES_WINDOW_CHECK = False
DATASET_PATH = PROJECT_ROOT / "data" / "Dataset.parquet"

if not RUN_OPTIONAL_SALES_WINDOW_CHECK:
    print(
        "Optional sales-window check skipped. "
        "Set RUN_OPTIONAL_SALES_WINDOW_CHECK = True to run it manually."
    )
elif not DATASET_PATH.exists():
    print(f"Dataset.parquet not found at {DATASET_PATH}; optional sales-window check skipped.")
else:
    sales_cols = ["TimeID", "NettoKr"]
    sales = pd.read_parquet(DATASET_PATH, columns=sales_cols)
    sales["hour"] = pd.to_numeric(sales["TimeID"], errors="coerce").astype("Int64")
    sales["NettoKr"] = pd.to_numeric(sales["NettoKr"], errors="coerce").fillna(0.0)
    by_hour = sales.groupby("hour", observed=True)["NettoKr"].sum().sort_index()
    total_sales = float(by_hour.sum())
    rows = []
    for label, start, end in [
        ("trade_08_18", 8, 18),
        ("trade_08_19", 8, 19),
        ("trade_10_18", 10, 18),
    ]:
        share = float(by_hour.loc[(by_hour.index >= start) & (by_hour.index <= end)].sum() / total_sales)
        rows.append({"window": label, "sales_share": share})
    ranked = by_hour.sort_values(ascending=False)
    cumulative = ranked.cumsum() / total_sales
    selected_hours = []
    for hour, share in cumulative.items():
        selected_hours.append(int(hour))
        if share >= 0.95:
            break
    print("Sales share by candidate trading window:")
    display(pd.DataFrame(rows).round(4))
    print(f"Hours needed to cover approximately 95% of sales: {sorted(selected_hours)}")
    del sales
    gc.collect()


## Manual run notes

When this notebook is run, execute it from the top through validation and confirm that `realised_weather_daily.parquet` and `realised_weather_daily_windows.parquet` are written under `data/processed/`. Inspect row counts, 46-store coverage, humidity/cloud bounds, non-negative precipitation, sentinel guards, and the local full-day versus UTC-day comparison.

The parser processes one weather variable at a time and deletes hourly frames after aggregation. Later documentation updates should record actual row counts, file sizes, date ranges, coverage summaries, validation results, and the realised-window schema. Realised weather remains a MET Nordic Analysis proxy and must not be used as operational M2/M4 input.
